In [1]:
import numpy as np
from PIL import Image
from scipy.ndimage import label
import os


#1. Load image 
def load_mask(path):
    img = Image.open(path).convert("L")
    mask = np.array(img)
    mask = (mask > 0).astype(np.uint8)  
    return mask


#2. Keep largest component 
def largest_component(mask):
    labeled, num = label(mask)

    if num == 0:
        return mask

    sizes = [(labeled == i).sum() for i in range(1, num + 1)]
    largest = np.argmax(sizes) + 1

    return (labeled == largest).astype(np.uint8)


#3. Crop to object
def crop_to_object(mask):
    ys, xs = np.where(mask == 1)

    if len(ys) == 0:
        return mask

    return mask[ys.min():ys.max()+1, xs.min():xs.max()+1]


#4. Split by centroid
def split_mask(mask):
    ys, xs = np.where(mask == 1)
    cy = int(np.mean(ys))
    cx = int(np.mean(xs))

    left  = mask[:, :cx]
    right = mask[:, cx:]

    top    = mask[:cy, :]
    bottom = mask[cy:, :]

    return left, right, top, bottom


#5. Align shapes (simple safe version)
def align(A, B):
    h = min(A.shape[0], B.shape[0])
    w = min(A.shape[1], B.shape[1])
    return A[:h, :w], B[:h, :w]


#6. Asymmetry score
def asymmetry(A, B):
    A = (A > 0).astype(np.uint8)
    B = (B > 0).astype(np.uint8)

    diff = np.abs(A.astype(int) - B.astype(int))

    return diff.sum() / (A.sum() + B.sum() + 1e-8)


#7. Main function 
def compute_asymmetry(path):
    mask = load_mask(path)
    mask = largest_component(mask)
    mask = crop_to_object(mask)

    left, right, top, bottom = split_mask(mask)

    # align
    left, right = align(left, right)
    top, bottom = align(top, bottom)

    # flip
    right = np.fliplr(right)
    bottom = np.flipud(bottom)

    # compute
    h = asymmetry(left, right)
    v = asymmetry(top, bottom)

    return {
        "horizontal": float(h),
        "vertical": float(v),
        "combined": float((h + v) / 2)
    }

folder = "data/masks"

for filename in os.listdir(folder):
    if filename.endswith(".png"):
        path = os.path.join(folder, filename)

        try:
            res = compute_asymmetry(path)

            print(f"{filename} -> H: {res['horizontal']:.3f}, "
                  f"V: {res['vertical']:.3f}, "
                  f"C: {res['combined']:.3f}")

        except Exception as e:
            print(f"Nope {filename}: {e}")


PAT_1546_1872_824_mask.png -> H: 0.057, V: 0.068, C: 0.063
PAT_851_4520_743_mask.png -> H: 0.691, V: 0.682, C: 0.686
PAT_1184_676_904_mask.png -> H: 0.141, V: 0.104, C: 0.122
PAT_286_1458_381_mask.png -> H: 0.284, V: 0.233, C: 0.259
PAT_355_732_625_mask.png -> H: 0.233, V: 0.120, C: 0.176
PAT_1746_3284_798_mask.png -> H: 0.265, V: 0.338, C: 0.302
PAT_2034_4266_880_mask.png -> H: 0.144, V: 0.144, C: 0.144
PAT_860_1641_440_mask.png -> H: 0.264, V: 0.046, C: 0.155
PAT_313_669_935_mask.png -> H: 0.119, V: 0.201, C: 0.160
PAT_53_82_657_mask.png -> H: 0.070, V: 0.081, C: 0.075
PAT_2051_4356_460_mask.png -> H: 0.166, V: 0.213, C: 0.190
PAT_1760_3326_525_mask.png -> H: 0.205, V: 0.195, C: 0.200
PAT_690_1309_981_mask.png -> H: 0.044, V: 0.029, C: 0.036
PAT_474_912_879_mask.png -> H: 0.147, V: 0.148, C: 0.148
PAT_1586_2623_989_mask.png -> H: 0.126, V: 0.064, C: 0.095
PAT_730_1385_585_mask.png -> H: 0.085, V: 0.077, C: 0.081
PAT_120_183_623_mask.png -> H: 0.105, V: 0.102, C: 0.103
PAT_224_343_782

KeyboardInterrupt: 